# Customer Support Context Engineering Demo

A complete demonstration of context engineering techniques using a realistic customer support scenario.

**Scenario**: Jane Doe, a Premium member, has issues with order #AC-7823 (delayed shipping, refund request) and wants to check order #AC-7901. She also wants to return an older order #AC-7654.

| Stage | Technique | Problem Solved |
|-------|-----------|----------------|
| 0 | Naive concatenation | Baseline — no budget control |
| 1 | Priority-based trimming | Fits budget, drops entire layers |
| 2 | Budget controller + thresholds | Adaptive, graduated |
| 3 | State tracking + compression | Industrial-grade, quality-preserving |

In [ ]:
import sys
import json
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from shared import (
    ContextLayer, ContextComposer, count_tokens,
    TokenBudgetController, BudgetConfig,
    AdaptiveCompressor,
)
from customer_support import (
    generate_support_scenario,
    SupportContextManager,
)

print("✓ Imports successful")

## Step 1: Load Scenario Data

Generate realistic customer support data including customer profile, orders, conversation history, and knowledge base.

In [ ]:
# Load scenario
scenario = generate_support_scenario()

# Convert to strings
system_prompt = f"""You are a helpful customer support agent for AcmeCorp.
Always be polite, accurate, and concise.
Verify order numbers before processing refunds.
Customer: {scenario.customer.name} ({scenario.customer.membership_tier.upper()} member)"""

knowledge_base = scenario.knowledge_base

# Split conversation history
split_point = max(0, len(scenario.conversation_history) - 4)
old_history = "\n".join(
    f"{turn.role.upper()}: {turn.content}"
    for turn in scenario.conversation_history[:split_point]
)
recent_history = "\n".join(
    f"{turn.role.upper()}: {turn.content}"
    for turn in scenario.conversation_history[split_point:]
)

user_query = scenario.current_query

# Token counts
token_counts = {
    "system_prompt": count_tokens(system_prompt),
    "knowledge_base": count_tokens(knowledge_base),
    "old_history": count_tokens(old_history),
    "recent_history": count_tokens(recent_history),
    "user_query": count_tokens(user_query),
}

print("Token counts:")
for name, t in token_counts.items():
    print(f"  {name:20s}: {t:4d}")
print(f"  {'TOTAL':20s}: {sum(token_counts.values())}")

## Stage 0: Naive Concatenation

No budget awareness. Just join everything together.

In [ ]:
BUDGET = 1_800  # Tight budget to force compression

naive_context = "\n\n".join([
    system_prompt,
    knowledge_base,
    old_history,
    recent_history,
    user_query,
])
naive_tokens = count_tokens(naive_context)

print(f"Stage 0: {naive_tokens} / {BUDGET} tokens ({naive_tokens/BUDGET:.1%})")
if naive_tokens > BUDGET:
    print(f"⚠️  OVER BUDGET by {naive_tokens - BUDGET} tokens")

## Stage 1: Priority-Based Trimming

Assign P0-P5 priorities. Drop lowest-priority layers first.

In [ ]:
composer = ContextComposer(total_budget=BUDGET, output_reserve_ratio=0.20)

layers = [
    ContextLayer("system_prompt", system_prompt, priority="p0"),
    ContextLayer("user_query", user_query, priority="p1"),
    ContextLayer("recent_history", recent_history, priority="p2"),
    ContextLayer("knowledge_base", knowledge_base, priority="p3"),
    ContextLayer("old_history", old_history, priority="p5"),
]

assembled = composer.compose(layers)

print(f"\nStage 1: {assembled.total_tokens} / {assembled.input_budget} tokens")
print(f"Utilization: {assembled.utilization:.1%}")
print(f"Dropped layers: {assembled.trimmed_layers}")

# Comparison
print("\n" + "═"*60)
print(f"  {'Metric':<25} {'Stage 0':>12} {'Stage 1':>12}")
print("═"*60)
print(f"  {'Tokens':<25} {naive_tokens:>12} {assembled.total_tokens:>12}")
print(f"  {'Utilization':<25} {naive_tokens/BUDGET:>11.1%} {assembled.utilization:>11.1%}")
print(f"  {'Layers dropped':<25} {'0':>12} {len(assembled.trimmed_layers):>12}")
print("═"*60)

## Stage 2: Budget Controller with Thresholds

Graduated compression at 70%/85%/95% thresholds.

In [ ]:
# Visualize three scenarios
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Budget Controller — Three Utilization Scenarios", fontsize=13, fontweight="bold")

scenarios = [("65% (OK)", 0.65), ("80% (Soft)", 0.80), ("92% (Hard)", 0.92)]
layer_names = ["sys", "kb", "old", "rec", "qry"]
colors = ["#4CAF50", "#2196F3", "#F44336", "#FF9800", "#00BCD4"]

for ax, (label, target_util) in zip(axes, scenarios):
    config = BudgetConfig(total_window=BUDGET, output_reserve_ratio=0.20)
    ctrl = TokenBudgetController(config)
    
    # Scale content to hit target utilization
    scale = target_util
    ctrl.add("system_prompt", system_prompt, compressible=False, priority=0)
    ctrl.add("user_query", user_query, compressible=False, priority=1)
    ctrl.add("knowledge_base", knowledge_base[:int(len(knowledge_base)*scale)], compressible=True, priority=3)
    ctrl.add("recent_history", recent_history, compressible=True, priority=2)
    ctrl.add("old_history", old_history[:int(len(old_history)*scale)], compressible=True, priority=5)
    
    level = ctrl.compression_level()
    report = ctrl.report()
    
    token_vals = [
        report["layers"].get(n, {}).get("tokens", 0)
        for n in ["system_prompt", "knowledge_base", "old_history", "recent_history", "user_query"]
    ]
    
    ax.bar(range(5), token_vals, color=colors, alpha=0.85)
    ax.set_xticks(range(5))
    ax.set_xticklabels(layer_names, fontsize=9)
    ax.set_ylabel("Tokens")
    ax.set_title(f"{label}\nLevel: {level.upper()}", fontsize=10)
    ax.set_ylim(0, max(token_vals) * 1.3)
    
    # Add utilization text
    total = sum(token_vals)
    util = total / config.input_budget
    ax.text(0.98, 0.95, f"{util:.1%}", transform=ax.transAxes,
            ha='right', va='top', fontsize=11, fontweight='bold')

# Legend
legend_patches = [mpatches.Patch(color=colors[i], label=layer_names[i]) for i in range(5)]
fig.legend(handles=legend_patches, loc='lower center', ncol=5, fontsize=9, bbox_to_anchor=(0.5, -0.02))
plt.tight_layout()
plt.show()

## Stage 3: State Tracking + Compression

Industrial-grade with:
- Schema-driven state tracking (order ID, refund status)
- Adaptive compression per layer type
- Multi-turn context management

In [ ]:
# Initialize manager
manager = SupportContextManager(total_budget=BUDGET, output_reserve_ratio=0.20)

# Update state with extracted information
manager.update_state(
    customer_id=scenario.customer.customer_id,
    active_order_id="AC-7823",
    refund_requested=True,
    refund_amount=89.00,
)

# Build context with state tracking
history_tuples = [(turn.role, turn.content) for turn in scenario.conversation_history]

final_context = manager.build_context(
    system_prompt=system_prompt,
    knowledge_base=knowledge_base,
    history=history_tuples,
    user_query=user_query,
    compress=True,
)

print("\n" + "─"*60)
print("Final Context Preview (first 500 chars):")
print("─"*60)
print(final_context[:500])
print("...")

In [ ]:
# Show state
print("\n" + "═"*60)
print("Session State:")
print("═"*60)
report = manager.get_report()
print(json.dumps(report, indent=2))

## Summary

| Stage | Technique | Tokens | Layers Dropped | Info Quality |
|-------|-----------|--------|----------------|--------------|
| 0 | Naive concatenation | Over budget | Uncontrolled | Unpredictable |
| 1 | Priority trimming | ≤ budget | 1 dropped | Low (binary) |
| 2 | Budget controller | ≤ budget | Graduated | Medium |
| 3 | State + Compression | ≤ budget | 0 dropped | **High** |

**Key insight**: Stage 3 fits the same budget but retains ALL information through intelligent compression and state tracking.